In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd
import os
import json

# For OpenAlex API
import requests
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders, config
config.api_key = "<YOUR_API_KEY>"

# GROBID Web Service Mode

We use the GROBID tool to extract structured information from ESRF experiment report PDFs, specifically the abstract and the references (DOIs). The extracted abstract can be added to the 'official' proposal abstract to provide more information for the OpenAlex model to ingest.

The ESRF experiment report PDFs can be found here:\
http://ftp.esrf.fr/pub/UserReports/ \

Unfortunately, the PDFs are too large to upload to GitHub, so I am unable to provide the exact set of PDFs (downloaded in May 2025) that I used.

GROBID output schema \
https://grobid.readthedocs.io/en/latest/training/Bibliographical-references/ \

GROBID will take quite some time to run for the configuration below. First, GROBID has to process the full document, and then make use of the Crossref API service to get DOIs for the references it extracts from the PDF. It is recommended to split the input PDFs into multiple batches and process them batch by batch rather than doing it all at once.

You can also access the extracted information in the GROBID_Web_Services_results.zip file in the Datasets/GROBID_Web_Services_results folder in the repository.

In [ ]:
# Set up and run GROBID server
from grobid_client.grobid_client import GrobidClient
config_file = "/Users/Folderpath/Code/config.json"
client = GrobidClient(config_path=config_file)

GROBID server is up and running


In [ ]:
# Process PDFs using GROBID; consolidate_citations enables use of the Crossref API service to match the references, improving extraction of citation information including DOIs
input_folderpath = "/Users/YourUser/ESRF_PDF_Documents/" # <--- INSERT YOUR INPUT FOLDER PATH HERE
output_folderpath = "/Users/YourUser/Output/" # <--- INSERT YOUR ONPUT FOLDER PATH HERE
client.process("processFulltextDocument", input_folderpath, consolidate_citations=True, n=1, include_raw_citations=True,output=output_folderpath)

Note that not all PDFs may be processed successfully. Reasons include corrupted PDFs or the PDFs being scans of a physical document. The latter results in empty .txt files instead of .tei.xml files.

# Processing

## Get list of DOIs for each PDF

In [21]:
# Function to extract relevant information from the GROBID XML output to a dictionary
def xml_to_dict(root):

    # Initalise empty dict
    pdf_dict={'referenced_works_doi':None, 'text':None}

    for listBibl in root.iter('{http://www.tei-c.org/ns/1.0}listBibl'):
        doi_ls=[]
        for ref in listBibl.iter('{http://www.tei-c.org/ns/1.0}biblStruct'):
            for idno in ref.iter('{http://www.tei-c.org/ns/1.0}idno'):
                if idno.attrib.get('type') == 'DOI':
                    doi_ls.append(idno.text)              

    whole_text=''
    for body in root.iter('{http://www.tei-c.org/ns/1.0}body'):
        # whole_text += ''.join(body.itertext())
        for p in body.iter('{http://www.tei-c.org/ns/1.0}p'):
            text=''.join(p.itertext())
            whole_text += text

    # for body in root.iter('{http://www.tei-c.org/ns/1.0}body'):
    #     text_ls=[]
    #     for p in body.iter('{http://www.tei-c.org/ns/1.0}p'):
    #         text_ls.append(p.text)

    pdf_dict['referenced_works_doi'] = doi_ls
    pdf_dict['text'] = whole_text
    
    return pdf_dict

In [ ]:
xml_folder = '{insert pathname}/GROBID_results_api_service'  # <--- INSERT YOUR FOLDER PATH HERE
pdf_dict={}

for xml_file in os.listdir(xml_folder):
    if xml_file.endswith('.tei.xml'):
        full_path = os.path.join(xml_folder, xml_file)
        file_name = xml_file.replace('.grobid.tei.xml', '')

        tree = ET.parse(full_path)
        root = tree.getroot()

        pdf_dict[file_name] = xml_to_dict(root)

In [6]:
len(pdf_dict)

12724

## Get OpenAlex IDs

In [8]:
# Function to get corresponding OpenAlex ID for each DOI
def doi_to_openalex(doi):
    doi = 'https://doi.org/' + doi if not doi.startswith('https://doi.org/') else doi
    try:
        work = Works()[doi]
        openalex_url = work['id']
        openalex_id = openalex_url.split('/')[-1]  # Extract the OpenAlex ID from the URL
        return openalex_id
    except Exception as e:
        return None

In [ ]:
# Loop through the PDF dictionary and add OpenAlex IDs for each DOI

for pdf in pdf_dict:
    doi_ls = pdf_dict[pdf]['DOI']
    openalex_ls = []
    for doi in doi_ls:
        openalex_id = doi_to_openalex(doi)
        if openalex_id:
            openalex_ls.append(openalex_id)
    pdf_dict[pdf]['OpenAlex'] = openalex_ls

In [32]:
pdf_dict['24250_D']

{'referenced_works_doi': ['10.7554/elife.40712.023',
  '10.1002/0471701343.sdp20625',
  '10.1002/0471701343.sdp20625',
  '10.53738/revmed.2018.14.588-89.0090'],
 'text': '',
 'openalex_ids': ['W4250897165', 'W4250897165', 'W4211103379']}

In [ ]:
filepath = '{insert pathname}/PDF_metadata.json'  # <--- INSERT YOUR FILEPATH HERE
with open(filepath, 'w') as f:
    json.dump(pdf_dict, f, indent=2)
